# Held-out scroll head-to-head: pherc0841 and pherc0009b

Loads two trained models and renders both held-out scrolls at full extent with each, using the same
figure as the test-fragment viewer (`test_inference.ipynb`), 6 columns x 2 rows:

| depth estimate | depth estimate +2 | depth estimate -2 | max of (0, +2, -2) | depth estimate + inklabel overlay | 2.4um inklabel |
|---|---|---|---|---|---|
| **d=0→8** | **d=4→12** | **d=8→16** | **d=12→20** | **d=20→28** | **max of fixed windows** |

- each model replays its own saved run config (field size, depth window, architecture), so models with different geometry can be compared
- an entry with several members is an ensemble: every member renders the scroll and the probabilities are averaged
- checkpoints default to `final.pth`, the weights the training-time full render (`R_M/...`) scored
- each render prints recall at 1% / 5% FPR and pAUC@1% for the depth-estimate prediction over the scroll's Train / Valid regions (`train_masks/<id>.png`) and the whole scroll
- the overlay paints the run's training label set (`inklabel_dir`, normally `dilated_inklabels`) onto the depth-estimate prediction as in `eval_scrolls.ipynb` (label cells brightened, the rest dimmed) with the Train / Valid split dotted in white; the last panel is `inklabels/2_4um/<id>.png`, falling back to `inklabels/1_1um/`, then `inklabels/<id>.png`
- the volume of one scroll is kept in RAM and shared by both models; switching scroll releases it
- run the setup cells, then the four comparison cells at the end
- figures are saved to `output/holdout_eval/<model label>/<scroll>.jpg`

In [ ]:
# the two entries to compare; an entry with several members is a probability-averaged ensemble
from pathlib import Path

RUNS = Path("/vesuvius/runs_archs36")
NATIVE96 = {"run_dir": RUNS / "36_holdout_native96", "checkpoint": "final.pth"}
DUAL192_SOFTER = {"run_dir": RUNS / "36_holdout_dual_scale_softer_28_08-07-51", "checkpoint": "final.pth"}
# the same 192 dual recipe at edge sigma 4: the closest finished replicate of DUAL192_SOFTER
DUAL192 = {"run_dir": RUNS / "36_holdout_dual_scale_27_11-24-50", "checkpoint": "final.pth"}
MODEL_SPECS = {
    "A": {"label": "c36_native96", "members": [NATIVE96]},
    "B": {"label": "ensemble_n96_dual192softer_dual192", "members": [NATIVE96, DUAL192_SOFTER, DUAL192]},
}
HOLDOUTS = {"pherc0841": 20260221022814, "pherc0009b": 20250919125754}
# inference window step in px (<= the 64 px multitile output); 32 matches the training-time render
EVAL_STRIDE = 32
SURFACE_SHIFTS = (0, 2, -2)
FIXED_WINDOW_STARTS = (0, 4, 8, 12, 20)
MASK_CROP_MARGIN = 32
PAD_PX = 24
OUTPUT_DIR = "output/holdout_eval"

In [ ]:
import copy
import gc
import importlib
import json
import os
import sys
import types

REPO = "/vesuvius"
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import zarr
from IPython.display import display
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

from utils.config import Config, ScrollConfig
from utils.dataloader import DataManager, _load_unified_cache
from utils.model import create_model
from utils.platform import get_zarr_dir
from utils import visualizer as visualizer_module

# reload so an existing kernel cannot retain a stale inference reader
visualizer_module = importlib.reload(visualizer_module)
TBV = visualizer_module.TensorboardVisualizer
group_by_depth = visualizer_module.group_by_depth
predict_tiles = visualizer_module.predict_tiles

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# per-model config and checkpoint loading
_AUXILIARY_PREFIXES = ("supcon_head.", "domain_head.")


def build_config(run_config):
    config = Config()
    # replay the full saved run config so geometry and surface window match training
    for name, value in run_config["data"].items():
        if name == "scrolls":
            config.data.scrolls = [ScrollConfig(**item) for item in value]
        elif name != "probe_rois" and hasattr(config.data, name):
            setattr(config.data, name, value)
    for name, value in run_config["model"].items():
        if hasattr(config.model, name):
            setattr(config.model, name, value)
    config.data.eval_stride = EVAL_STRIDE
    config.data.zarr_path = get_zarr_dir()
    config.model.compile_model = False
    config.tra.supcon = False
    config.tra.dann = False
    config.device = "cuda" if torch.cuda.is_available() else "cpu"
    return config


def _checkpoint_state(path):
    state = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    cleaned = {}
    for key, value in state.items():
        key = key.removeprefix("module.").removeprefix("_orig_mod.")
        if not key.startswith(_AUXILIARY_PREFIXES):
            cleaned[key] = value
    return cleaned, len(state) - len(cleaned)


_LOADED = {}


def load_member(member):
    """one checkpoint with its own run config; shared by every entry that lists it"""
    key = (str(member["run_dir"]), member["checkpoint"])
    if key in _LOADED:
        return _LOADED[key]
    run_config = json.loads((member["run_dir"] / "config.json").read_text(encoding="utf-8"))
    checkpoint = Path(REPO) / run_config["model_dir"] / member["checkpoint"]
    if not checkpoint.is_file():
        raise FileNotFoundError(f"checkpoint is missing: {checkpoint}")
    config = build_config(run_config)
    model, parameter_count = create_model(config)
    state, ignored = _checkpoint_state(checkpoint)
    model.load_state_dict(state, strict=True)
    model.eval()
    print(
        f"[load] {checkpoint} params={parameter_count:,} ignored={ignored} "
        f"ctx={config.data.context_size}/ds{config.data.context_downsample} depth={config.data.depth} "
        f"dual_scale={config.model.dual_scale}"
    )
    _LOADED[key] = {"name": member["run_dir"].name, "config": config, "model": model}
    return _LOADED[key]


def load_entry(spec):
    members = [load_member(member) for member in spec["members"]]
    depths = {member["config"].data.depth for member in members}
    if len(depths) != 1:
        raise ValueError(f"{spec['label']}: ensemble members need one depth window, got {sorted(depths)}")
    return {"label": spec["label"], "members": members, "config": members[0]["config"]}

In [ ]:
# shared scroll inputs, inference passes, labels, and the 6x2 figure
PRED_CMAP = plt.get_cmap("inferno").copy()
PRED_CMAP.set_bad((0.45, 0.45, 0.45))
# matplotlib/agg and jpeg both cap each image dimension below 2**16 px
MAX_FIGURE_PX = 65000
PREVIEW_DOWNSCALE = 3
FIGURE_DPI = 100
_SCROLL_CACHE = {}


def _mask_bbox(mask_bool, margin, align):
    ys, xs = np.where(mask_bool)
    if ys.size == 0:
        return 0, mask_bool.shape[0], 0, mask_bool.shape[1]
    y0 = max(0, ((int(ys.min()) - margin) // align) * align)
    x0 = max(0, ((int(xs.min()) - margin) // align) * align)
    y1 = min(mask_bool.shape[0], ((int(ys.max()) + 1 + margin + align - 1) // align) * align)
    x1 = min(mask_bool.shape[1], ((int(xs.max()) + 1 + margin + align - 1) // align) * align)
    return y0, y1, x0, x1


def scroll_inputs(scroll_id, config):
    """volume crop, mask, norm and surface maps for one scroll; only the latest scroll stays in RAM"""
    if scroll_id not in _SCROLL_CACHE:
        _SCROLL_CACHE.clear()
        gc.collect()
        mask_full = np.array(Image.open(f"masks/{scroll_id}.png").convert("L")) > 0
        bounds = _mask_bbox(mask_full, MASK_CROP_MARGIN, int(config.data.tile_size))
        y0, y1, x0, x1 = bounds
        source = zarr.open(os.path.join(config.data.zarr_path, f"{scroll_id}.zarr"), mode="r")
        print(f"[preload] {scroll_id} bbox={bounds} -> RAM", flush=True)
        stats = _load_unified_cache()[str(scroll_id)]
        surface_dir = os.path.join(config.data.surface_label_dir, str(scroll_id))
        _SCROLL_CACHE[scroll_id] = {
            "bounds": bounds,
            "mask_shape": mask_full.shape,
            "vol": np.asarray(source[:, y0:y1, x0:x1]),
            "mask": mask_full[y0:y1, x0:x1],
            "norm": (stats["mean"], stats["std"], stats["min"], stats["max"]),
            "surface_depth": np.load(os.path.join(surface_dir, "depth.npy"), mmap_mode="r")[y0:y1, x0:x1],
            "surface_confidence": np.load(os.path.join(surface_dir, "confidence.npy"), mmap_mode="r")[y0:y1, x0:x1],
        }
    return _SCROLL_CACHE[scroll_id]


def _predict_member(member, scroll_id, inputs, surface_shift, depth_start):
    config = member["config"]
    height, width = inputs["mask"].shape
    fake = types.SimpleNamespace(c=config)
    coords = TBV._gen_tile_coords(
        fake, (config.data.d_start, config.data.d_end), (0, height), (0, width), inputs["mask"],
        z_step=config.data.depth,
    )
    grouped = group_by_depth(coords)
    if len(grouped) != 1:
        raise RuntimeError(f"surface-relative inference expected one depth pass, got {len(grouped)}")
    depth_offset, coords = next(iter(grouped.items()))
    if depth_start is None:
        pass_config, start, tag = config, config.data.d_start + depth_offset, f"surface{surface_shift:+d}"
    else:
        pass_config = copy.deepcopy(config)
        pass_config.data.surface_relative_depth_window = False
        # a deeper window cannot start past the last full stack of slices
        start = min(int(depth_start), int(inputs["vol"].shape[0]) - config.data.depth)
        tag = f"d{start}-{start + config.data.depth}"
    return predict_tiles(
        pass_config, member["model"], inputs["vol"], inputs["mask"], coords, (0, height), (0, width),
        start, f"holdout_{scroll_id}_{tag}", *inputs["norm"],
        surface_depth_map=inputs["surface_depth"],
        surface_confidence_map=inputs["surface_confidence"],
        surface_depth_offset=surface_shift,
    )


def predict_map(entry, scroll_id, inputs, surface_shift=0, depth_start=None):
    """surface-relative pass shifted by surface_shift, or a fixed window when depth_start is set;
    ensembles average their members' probabilities"""
    maps = [_predict_member(m, scroll_id, inputs, surface_shift, depth_start) for m in entry["members"]]
    if len(maps) == 1:
        return maps[0]
    height = min(pmap.shape[0] for pmap in maps)
    width = min(pmap.shape[1] for pmap in maps)
    return np.nanmean(np.stack([pmap[:height, :width] for pmap in maps]), axis=0)


def label_grids(entry, scroll_id, inputs, grid_shape):
    """the run's training label, valid cells and train-region cells on the prediction grid"""
    config = entry["config"]
    unit = int(config.model.multitile_subtile)
    y0, y1, x0, x1 = inputs["bounds"]
    label_dir = getattr(config.data, "inklabel_dir", None) or "./inklabels"
    labels = np.array(Image.open(os.path.join(label_dir, f"{scroll_id}.png")).convert("L"))[y0:y1, x0:x1] > 127
    fake = types.SimpleNamespace(c=config)
    height, width = inputs["mask"].shape
    label_binary, _, valid = TBV._compute_tile_maps(
        fake, labels.astype(np.float32), inputs["mask"], (0, height), (0, width), tile=unit,
    )
    grid_h, grid_w = min(grid_shape[0], valid.shape[0]), min(grid_shape[1], valid.shape[1])
    label_binary, valid = label_binary[:grid_h, :grid_w], valid[:grid_h, :grid_w]
    train_grid = None
    train_mask = cv2.imread(os.path.join(config.data.train_mask_dir, f"{scroll_id}.png"), cv2.IMREAD_GRAYSCALE)
    if train_mask is not None:
        aligned = DataManager._align_manual_mask(train_mask, unit)[y0:y0 + grid_h * unit, x0:x0 + grid_w * unit]
        aligned = np.pad(aligned, ((0, grid_h * unit - aligned.shape[0]), (0, grid_w * unit - aligned.shape[1])))
        train_grid = aligned.reshape(grid_h, unit, grid_w, unit).all(axis=(1, 3))
    return label_binary, valid, train_grid


def print_metrics(entry, name, scroll_id, inputs, pred):
    """the training-time full-render recall / pAUC metrics for one prediction map"""
    label_binary, valid, train_grid = label_grids(entry, scroll_id, inputs, pred.shape)
    pred = pred[:valid.shape[0], :valid.shape[1]]
    regions = {"all": valid}
    if train_grid is not None:
        regions = {"Train": valid & train_grid, "Valid": valid & ~train_grid, **regions}
    fake = types.SimpleNamespace(c=entry["config"])
    parts = []
    for region, keep in regions.items():
        keep = keep & np.isfinite(pred)
        scores, targets = pred[keep], label_binary[keep].astype(int)
        r1, pauc1 = TBV._compute_low_fpr_metrics(fake, scores, targets, max_fpr=0.01)
        r5, _ = TBV._compute_low_fpr_metrics(fake, scores, targets, max_fpr=0.05)
        parts.append(f"{region}: R@1%={r1:.3f} R@5%={r5:.3f} pAUC@1%={pauc1:.3f}")
    print(f"[metrics] {entry['label']} {name} | " + " | ".join(parts))


def _nan_max(maps):
    return np.fmax.reduce(np.stack(maps), axis=0)


def _crop_scaled(image, inputs):
    """crop a label drawn on any grid proportional to the mask to the inference bounds"""
    y0, y1, x0, x1 = inputs["bounds"]
    mask_h, mask_w = inputs["mask_shape"]
    img_h, img_w = image.shape
    return image[
        int(y0 * img_h / mask_h):int(y1 * img_h / mask_h),
        int(x0 * img_w / mask_w):int(x1 * img_w / mask_w),
    ]


def overlay_rgb(pred, label_binary):
    """eval_scrolls overlay: label cells blended halfway to white, other cells dimmed 15%"""
    rgb = PRED_CMAP(np.clip(np.nan_to_num(pred, nan=0.0), 0.0, 1.0))[..., :3].copy()
    h, w = min(label_binary.shape[0], rgb.shape[0]), min(label_binary.shape[1], rgb.shape[1])
    ink = label_binary[:h, :w]
    region = rgb[:h, :w]
    region[~ink] *= 0.85
    region[ink] = 0.5 * region[ink] + 0.5
    rgb[~np.isfinite(pred)] = (0.45, 0.45, 0.45)
    return rgb


def reference_label(scroll_id, inputs):
    for path in (f"inklabels/2_4um/{scroll_id}.png", f"inklabels/1_1um/{scroll_id}.png", f"inklabels/{scroll_id}.png"):
        if os.path.exists(path):
            image = np.array(Image.open(path).convert("L"))
            # 0/1 masks become visible 0/255
            if image.max() <= 1:
                image = image * 255
            crop_h = inputs["bounds"][1] - inputs["bounds"][0]
            crop_w = inputs["bounds"][3] - inputs["bounds"][2]
            crop = cv2.resize(_crop_scaled(image, inputs), (crop_w, crop_h), interpolation=cv2.INTER_AREA)
            return crop, os.path.dirname(path)
    raise FileNotFoundError(f"no ink label for {scroll_id}")


def _save_figure(rows, crop_hw, out_path):
    """lay out (title, image, kind[, overlay]) rows on a pixel-exact grid and save it"""
    crop_height, crop_width = crop_hw
    n_rows, n_cols = len(rows), max(len(row) for row in rows)
    gap = max(PAD_PX, crop_width // 40)
    title_h = max(60, crop_height // 14)
    fig_w = n_cols * crop_width + (n_cols + 1) * gap
    fig_h = n_rows * (title_h + crop_height) + (n_rows + 1) * gap
    scale = min(1.0, MAX_FIGURE_PX / fig_w, MAX_FIGURE_PX / fig_h)
    panel_hw = (max(1, int(crop_height * scale)), max(1, int(crop_width * scale)))
    title_pt = title_h * scale * 0.45 * 72 / FIGURE_DPI
    fig = plt.figure(figsize=(fig_w * scale / FIGURE_DPI, fig_h * scale / FIGURE_DPI), dpi=FIGURE_DPI, facecolor="white")
    for r, row in enumerate(rows):
        row_top = gap + r * (title_h + crop_height + gap) + title_h
        for c, (title, image, kind, *extra) in enumerate(row):
            ax = fig.add_axes([
                (gap + c * (crop_width + gap)) / fig_w,
                1 - (row_top + crop_height) / fig_h,
                crop_width / fig_w,
                crop_height / fig_h,
            ])
            if kind == "label":
                small = cv2.resize(image, (panel_hw[1], panel_hw[0]), interpolation=cv2.INTER_AREA)
                ax.imshow(small, cmap="gray", vmin=0, vmax=255, interpolation="nearest", aspect="auto")
            elif kind == "overlay":
                label_binary, train_grid = extra
                ax.imshow(overlay_rgb(image, label_binary), interpolation="nearest", aspect="auto")
                if train_grid is not None and train_grid.any() and not train_grid.all():
                    # line width in points scales with the panel so the split stays visible
                    ax.contour(train_grid.astype(np.float32), levels=[0.5], colors="white",
                               linewidths=max(1.0, panel_hw[1] / 500), linestyles=":")
            else:
                ax.imshow(np.ma.masked_invalid(image), cmap=PRED_CMAP, vmin=0, vmax=1,
                          interpolation="nearest", aspect="auto")
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(title, fontsize=title_pt, pad=title_pt * 0.4)
    fig.savefig(out_path, dpi=FIGURE_DPI, facecolor="white", pil_kwargs={"quality": 92})
    plt.close(fig)


def render_holdout(name, entry):
    scroll_id = HOLDOUTS[name]
    inputs = scroll_inputs(scroll_id, entry["config"])
    y0, y1, x0, x1 = inputs["bounds"]
    crop_hw = (y1 - y0, x1 - x0)
    depth = entry["config"].data.depth
    surface_maps = [predict_map(entry, scroll_id, inputs, surface_shift=shift) for shift in SURFACE_SHIFTS]
    fixed_maps = [predict_map(entry, scroll_id, inputs, depth_start=start) for start in FIXED_WINDOW_STARTS]
    print_metrics(entry, name, scroll_id, inputs, surface_maps[0])
    label_binary, _, train_grid = label_grids(entry, scroll_id, inputs, surface_maps[0].shape)
    reference, reference_source = reference_label(scroll_id, inputs)
    surface_titles = [f"{entry['label']}: prediction with depth estimate"] + [
        f"depth estimate shifted {shift:+d}" for shift in SURFACE_SHIFTS[1:]
    ]
    rows = [
        [(title, pmap, "pred") for title, pmap in zip(surface_titles, surface_maps)]
        + [
            ("max of depth estimates (0, +2, -2)", _nan_max(surface_maps), "pred"),
            ("depth estimate + inklabel overlay", surface_maps[0], "overlay", label_binary, train_grid),
            (f"inklabel ({reference_source})", reference, "label"),
        ],
        [(f"prediction d={start}→{start + depth}", pmap, "pred") for start, pmap in zip(FIXED_WINDOW_STARTS, fixed_maps)]
        + [("max of fixed-window predictions", _nan_max(fixed_maps), "pred")],
    ]
    out_dir = os.path.join(OUTPUT_DIR, entry["label"])
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{name}_{scroll_id}.jpg")
    _save_figure(rows, crop_hw, out_path)
    print(f"[saved] {out_path} bbox={inputs['bounds']}")
    with Image.open(out_path) as saved:
        display(saved.resize(
            (max(1, saved.width // PREVIEW_DOWNSCALE), max(1, saved.height // PREVIEW_DOWNSCALE)),
            Image.LANCZOS,
        ))
    del rows, surface_maps, fixed_maps
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# load every checkpoint once (a member shared by both entries is loaded once)
MODELS = {key: load_entry(spec) for key, spec in MODEL_SPECS.items()}

In [ ]:
# pherc0841 - model A
render_holdout("pherc0841", MODELS["A"])

In [ ]:
# pherc0841 - model B
render_holdout("pherc0841", MODELS["B"])

In [ ]:
# pherc0009b - model A
render_holdout("pherc0009b", MODELS["A"])

In [ ]:
# pherc0009b - model B
render_holdout("pherc0009b", MODELS["B"])